<a href="https://colab.research.google.com/github/ammar-aa/Fly_rank_internship_repo/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
"""
One row = one content item, for one client, for one calendar month.
this notebook's window is March 2026 (month=2026-03), restricted to clients whose gsc_data_start/ga4_data_start falls on or before that month.
"""

"\nOne row = one content item, for one client, for one calendar month.\nthis notebook's window is March 2026 (month=2026-03), restricted to clients whose gsc_data_start/ga4_data_start falls on or before that month.\n"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [11]:
"""
FEATURE (the five required for this week, per task scope):
- gsc_impressions, gsc_clicks -> build ctr
- gsc_avg_position
- ga4_engaged_sessions, ga4_sessions -> build engagement_rate
- scroll_events, ga4_sessions -> build scroll_rate
- trend_pct (gsc_impressions, current vs prior month; guarded to only compute
  when prior-month impressions >= 30, and clipped to +/-300% to control
  small-denominator noise) -> the gate. Applies whenever GSC is available
  (Tier 1 and GSC-only Tier 2), regardless of GA4 availability.

  days_since_last_update was dropped after verification showed
  content_updated_date is a current-state snapshot (as of the July 2026
  export), not a historical record -- 88.5% of March's content items show an
  update date AFTER March 31, which would leak future information into a
  March-dated staleness feature.

  A parallel engagement_trend_pct (GA4-based gate) was tried and dropped:
  even after guarding (>=5 prior engaged sessions) and clipping, only 788 of
  331,437 rows (~0.24%) had a usable value -- too sparse to function as a
  real feature at this month's actual GA4 volume, not a logic problem.

DEFERRED - DECIDED BUT NOT BUILT THIS WEEK (task caps Week 3 at five features):
- Tiered scoring (full / GSC-only / GA4-only / insufficient), reflecting real
  availability gaps found in March 2026 (~63% of rows have neither GSC nor GA4
  available; ~33% GSC-only; ~0.5% GA4-only; ~3.7% both)
- Balancing signals per tier (raw gsc_impressions, raw gsc_clicks, raw
  ga4_sessions, ga4_total_engagement_sec) to avoid thin, ratio-only scoring in
  partial-signal tiers
- These are reserve potential signals for the capstone, not part of this week's
  five-feature answer

LABEL / PROXY (not a stored field - the formula that combines the features above):
- Refresh-priority score: gated by trend_pct, scored by CTR/position/
  engagement/scroll, normalized by content_type/word_count, modified by
  search_volume/competition

CONTEXT (needed for the pipeline, never scored directly, no leakage check required):
- client_hash_id, content_hash_id -> join keys
- content_type, word_count -> normalizers
- search_volume, competition -> modifiers
- gsc_data_start, ga4_data_start, has_gsc_access, has_ga4_access (dim_clients)
  -> window validation only
- ga4_data_available, gsc_data_available -> availability filters; also determine
  which tier/formula a row falls into in the deferred tiered-scoring design

EXCLUDED (permanently, with reasons):
- fact_content_query_90d (whole table) -> window_start/window_end are fixed at
  2026-04-02 to 2026-06-30 for every row (confirmed by query), making this table
  usable only for the sealed June test month, never for March or any other
  working month; its query-diversity signals are also excluded to keep the
  feature set small, explainable, and less prone to overfitting
- provider_used, model_used -> production metadata, not a performance signal;
  risks the model learning "which AI wrote this" instead of "does it need refresh"
- is_published, is_deleted, is_active -> filter criteria applied before scoring,
  not features
- trend_direction / is_declining_label-equivalents -> downstream restatements of
  trend_pct; circular if used as features
"""

'\nFEATURE (the five required for this week, per task scope):\n- gsc_impressions, gsc_clicks -> build ctr\n- gsc_avg_position\n- ga4_engaged_sessions, ga4_sessions -> build engagement_rate\n- scroll_events, ga4_sessions -> build scroll_rate\n- trend_pct (gsc_impressions, current vs prior month) and engagement_trend_pct\n  (ga4_engaged_sessions, current vs prior month) -> the gate. days_since_last_update\n  was dropped after verification showed content_updated_date is a current-state\n  snapshot (as of the July 2026 export), not a historical record -- 88.5% of\n  March\'s content items show an update date AFTER March 31, which would leak\n  future information into a March-dated staleness feature. Tier 1 combines both\n  trend signals; the exact combination rule (AND/OR/average) is left open pending\n  later validation, with AND used as this week\'s working placeholder.\n\nDEFERRED - DECIDED BUT NOT BUILT THIS WEEK (task caps Week 3 at five features):\n- Tiered scoring (full / GSC-only /

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [3]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

In [4]:
daily_march_check = con.sql(f"""
    SELECT
        COUNT(*)              AS row_count,
        MIN(report_date)      AS min_date,
        MAX(report_date)      AS max_date,
        COUNT(DISTINCT content_hash_id) AS distinct_content,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
daily_march_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,min_date,max_date,distinct_content,distinct_clients
0,9841378,2026-03-01,2026-03-31,331437,55


In [5]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE AND gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS both_available_rows
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
""").df()
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,gsc_available_rows,both_available_rows
0,9841378,413966.0,3611061.0,364347.0


In [6]:
monthly_agg = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        DATE_TRUNC('month', report_date) AS month,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks)      AS gsc_clicks
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id, DATE_TRUNC('month', report_date)
""").df()
monthly_agg

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,client_hash_id,month,gsc_impressions,gsc_clicks
0,content_05597932fe4da067,client_73cda7b4e4f265ea,2026-03-01,57.0,0.0
1,content_7a105f548d9c6916,client_73cda7b4e4f265ea,2026-03-01,6523.0,7.0
2,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,2026-03-01,453.0,0.0
3,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2026-03-01,2770.0,16.0
4,content_bfd1e41c2af250c8,client_73cda7b4e4f265ea,2026-03-01,48.0,0.0
...,...,...,...,...,...
331432,content_6c3e028a91a52e27,client_4a18d1793d92fb84,2026-03-01,0.0,0.0
331433,content_15246f36b972664d,client_4a18d1793d92fb84,2026-03-01,0.0,0.0
331434,content_2e38254bab54aa4b,client_4a18d1793d92fb84,2026-03-01,0.0,0.0
331435,content_2713aadb9f2b4bb4,client_86ebc2f12c01f586,2026-03-01,0.0,0.0


In [7]:
grain_check = con.sql("""
    SELECT content_hash_id, client_hash_id, month, COUNT(*) AS n
    FROM monthly_agg
    GROUP BY content_hash_id, client_hash_id, month
    HAVING COUNT(*) > 1
""").df()

print(f"Aggregated rows: {len(monthly_agg)} | Duplicate groups (should be 0): {len(grain_check)}")

Aggregated rows: 331437 | Duplicate groups (should be 0): 0


In [8]:
BASE = "hf://datasets/FlyRank/internship-warehouse"

tier_breakdown = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 'tier1_both'
            WHEN gsc_data_available IS TRUE AND ga4_data_available IS NOT TRUE THEN 'tier2_gsc_only'
            WHEN gsc_data_available IS NOT TRUE AND ga4_data_available IS TRUE THEN 'tier2_ga4_only'
            ELSE 'tier3_neither'
        END AS tier,
        COUNT(*) AS row_count
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY tier
    ORDER BY row_count DESC
""").df()

tier_breakdown

,tier,row_count
0,tier3_neither,6180698
1,tier2_gsc_only,3246714
2,tier1_both,364347
3,tier2_ga4_only,49619


In [9]:
BASE = "hf://datasets/FlyRank/internship-warehouse"

window_check = con.sql(f"""
    SELECT
        MIN(window_start) AS min_start,
        MAX(window_start) AS max_start,
        MIN(window_end)   AS min_end,
        MAX(window_end)   AS max_end,
        COUNT(DISTINCT window_start) AS distinct_starts,
        COUNT(DISTINCT window_end)   AS distinct_ends
    FROM read_parquet('{BASE}/fact_content_query_90d.parquet')
""").df()

window_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_start,max_start,min_end,max_end,distinct_starts,distinct_ends
0,2026-04-02,2026-04-02,2026-06-30,2026-06-30,1,1


In [10]:
BASE = "hf://datasets/FlyRank/internship-warehouse"

update_date_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_content_rows,
        SUM(CASE WHEN c.content_updated_date > DATE '2026-03-31' THEN 1 ELSE 0 END) AS updated_after_march,
        SUM(CASE WHEN c.content_updated_date <= DATE '2026-03-31' THEN 1 ELSE 0 END) AS updated_on_or_before_march,
        SUM(CASE WHEN c.content_updated_date IS NULL THEN 1 ELSE 0 END) AS missing_update_date
    FROM (SELECT DISTINCT content_hash_id FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')) m
    LEFT JOIN read_parquet('{BASE}/dim_content.parquet') c
        ON m.content_hash_id = c.content_hash_id
""").df()

update_date_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_march_content_rows,updated_after_march,updated_on_or_before_march,missing_update_date
0,331437,293358.0,38079.0,0.0


In [12]:
BASE = "hf://datasets/FlyRank/internship-warehouse"

march_agg = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions,
        SUM(gsc_clicks)      AS gsc_clicks,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS gsc_avg_position,
        SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions END) AS ga4_engaged_sessions,
        SUM(CASE WHEN ga4_data_available THEN ga4_sessions END)         AS ga4_sessions,
        SUM(CASE WHEN ga4_data_available THEN scroll_events END)       AS scroll_events
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id
""").df()

print(len(march_agg))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

331437


In [13]:
feb_agg = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_prior,
        SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions END) AS ga4_engaged_sessions_prior
    FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-02/data_0.parquet')
    GROUP BY content_hash_id, client_hash_id
""").df()

print(len(feb_agg))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

321546


In [21]:
MIN_VOLUME_GSC = 30   # gsc_impressions_prior floor -- impressions are naturally large-scale
MIN_VOLUME_GA4 = 5    # ga4_engaged_sessions_prior floor -- engaged sessions are naturally small-scale
TREND_CLIP = 3.0       # cap trend_pct-style ratios at +/-300% to control extreme small-base spikes

feature_frame = con.sql(f"""
    SELECT
        m.content_hash_id,
        m.client_hash_id,

        -- feature 1: ctr
        CASE WHEN m.gsc_impressions > 0 THEN m.gsc_clicks * 1.0 / m.gsc_impressions END AS ctr,

        -- feature 2: gsc_avg_position (already averaged, excl. zeros)
        m.gsc_avg_position,

        -- feature 3: engagement_rate
        CASE WHEN m.ga4_sessions > 0 THEN m.ga4_engaged_sessions * 1.0 / m.ga4_sessions END AS engagement_rate,

        -- feature 4: scroll_rate
        CASE WHEN m.ga4_sessions > 0 THEN m.scroll_events * 1.0 / m.ga4_sessions END AS scroll_rate,

        -- feature 5a: trend_pct (GSC side), guarded + clipped
        CASE WHEN f.gsc_impressions_prior >= {MIN_VOLUME_GSC}
             THEN GREATEST(-{TREND_CLIP}, LEAST({TREND_CLIP},
                (m.gsc_impressions - f.gsc_impressions_prior) * 1.0 / f.gsc_impressions_prior))
        END AS trend_pct,

        -- feature 5b: engagement_trend_pct (GA4 side), own guard + clip
        CASE WHEN f.ga4_engaged_sessions_prior >= {MIN_VOLUME_GA4}
             THEN GREATEST(-{TREND_CLIP}, LEAST({TREND_CLIP},
                (m.ga4_engaged_sessions - f.ga4_engaged_sessions_prior) * 1.0 / f.ga4_engaged_sessions_prior))
        END AS engagement_trend_pct

    FROM march_agg m
    LEFT JOIN feb_agg f
        ON m.content_hash_id = f.content_hash_id
        AND m.client_hash_id = f.client_hash_id
""").df()

print(len(feature_frame))
feature_frame[['trend_pct', 'engagement_trend_pct']].describe()

331437


,trend_pct,engagement_trend_pct
count,97071.000000,788.000000
mean,0.602048,-0.236148
std,1.036638,0.681201
min,-1.000000,-1.000000
25%,-0.104037,-0.666667
50%,0.303030,-0.400000
75%,1.010878,0.000000
max,3.000000,3.000000


In [22]:

display(feature_frame.describe(include='all'),feature_frame.isna().sum())

,content_hash_id,client_hash_id,ctr,gsc_avg_position,engagement_rate,scroll_rate,trend_pct,engagement_trend_pct
count,331437,331437,176738.000000,175304.000000,90237.000000,90237.000000,97071.000000,788.000000
unique,331437,55,NaN,NaN,NaN,NaN,NaN,NaN
top,content_9e0c10cbc337953d,client_625b6439094e23e4,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,31887,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,NaN,0.004594,17.050555,0.025977,0.225258,0.602048,-0.236148
std,NaN,NaN,0.037760,18.333942,0.110565,0.342433,1.036638,0.681201
min,NaN,NaN,0.000000,0.101639,0.000000,0.000000,-1.000000,-1.000000
25%,NaN,NaN,0.000000,5.500000,0.000000,0.000000,-0.104037,-0.666667
50%,NaN,NaN,0.000000,9.000000,0.000000,0.000000,0.303030,-0.400000
75%,NaN,NaN,0.002158,22.000000,0.000000,0.333333,1.010878,0.000000


,0
content_hash_id,0
client_hash_id,0
ctr,154699
gsc_avg_position,156133
engagement_rate,241200
scroll_rate,241200
trend_pct,234366
engagement_trend_pct,330649


In [23]:
"""
FIVE FEATURES - knowable at the decision moment because:

1. ctr (gsc_clicks / gsc_impressions) -- built entirely from March's own daily
   GSC rows; no data outside the March window is used.
2. gsc_avg_position -- same: averaged only over March's own daily rows,
   excluding zero-value (no-data) days.
3. engagement_rate (ga4_engaged_sessions / ga4_sessions) -- built only from
   March's own daily rows, and only where ga4_data_available IS TRUE for that
   row, so placeholder zeros are never mistaken for real engagement.
4. scroll_rate (scroll_events / ga4_sessions) -- same GA4-availability guard
   as engagement_rate.
5. trend_pct (the gate) -- compares March's impressions to February's, both
   already-completed months as of the March decision point; guarded to
   >=30 prior-month impressions and clipped to +/-300% to prevent
   small-denominator noise from producing an unstable trend value.
"""

"\nFIVE FEATURES - knowable at the decision moment because:\n\n1. ctr (gsc_clicks / gsc_impressions) -- built entirely from March's own daily\n   GSC rows; no data outside the March window is used.\n2. gsc_avg_position -- same: averaged only over March's own daily rows,\n   excluding zero-value (no-data) days.\n3. engagement_rate (ga4_engaged_sessions / ga4_sessions) -- built only from\n   March's own daily rows, and only where ga4_data_available IS TRUE for that\n   row, so placeholder zeros are never mistaken for real engagement.\n4. scroll_rate (scroll_events / ga4_sessions) -- same GA4-availability guard\n   as engagement_rate.\n5. trend_pct (the gate) -- compares March's impressions to February's, both\n   already-completed months as of the March decision point; guarded to\n   >=30 prior-month impressions and clipped to +/-300% to prevent\n   small-denominator noise from producing an unstable trend value.\n"

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [9]:
"""
DATA LIMITS - what this data can never tell you:

1) True update history at any past point in time.
   content_updated_date in dim_content is a current-state snapshot (as of the
   July 2026 export), not a historical record. For March 2026, 88.5% of content
   items (293,358 of 331,437) show an update date AFTER March 31 -- confirmed by
   query. This means "days since last update, as of March" is only knowable for
   the remaining ~11.5% of content items; for the majority, the only recorded
   update date lies in that item's future relative to March, and using it would
   leak forward-looking information into a March-dated feature. This is why
   days_since_last_update was dropped from this week's gate entirely.

2) A complete engagement picture across the panel.
   GA4 data is available for only ~4.2% of March's 9,841,378 rows, and that
   availability is concentrated unevenly: only 41 of 55 active March clients
   have any GA4-available rows at all, and among those 41, per-client coverage
   ranges from ~3% to ~35% of their March days (confirmed by query) -- reflecting
   staggered GA4 onboarding (ga4_data_start), not random missingness. ~63% of
   March rows have neither GSC nor GA4 available and cannot be scored at all.

3) Anything about periods outside a table's actual window.
   fact_content_query_90d's window_start/window_end are fixed at 2026-04-02 to
   2026-06-30 for every single row (confirmed by query: distinct_starts=1,
   distinct_ends=1). It is a single snapshot, not a repeating per-month table --
   it can describe the sealed June test month, but it can never validly describe
   March or any other earlier month, which is why it is excluded from this
   week's feature set entirely.

Together: this data can tell you a page's recent search/engagement trend and its
current-state attributes, but it cannot reliably reconstruct a page's true edit
history or full engagement picture at any earlier point in the panel -- these
are permanent, structural gaps in the release itself, not something more queries
or feature engineering can recover.
"""

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.